# Protein ↔ BiologicalProcess Relation-Wise Merge

Merges Protein–BiologicalProcess triples from CKG, CrossBAR, Monarch and TARKG;
fills missing protein head names from UniProt; deduplicates by `(head, relation, tail)`;
and saves the result.


## 0. Configuration

In [1]:
import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_BIOLOGICALPROCESS/ALL_PROTEIN_BIOLOGICALPROCESS.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Protein Lookup Dictionary — UniProt

In [2]:
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot
print(f"UniProt AC→Name dict: {len(uniprot_trembl_AC_Name_dict):,} entries")

UniProt AC→Name dict: 252,635,149 entries


## 2. Load KG Sources

### 2a. CKG

In [3]:
CKG_P_BP = pd.read_csv(PROC_DIR + 'CKG/File_12_Protein_BiologicalProcess_CKG.csv')
CKG_P_BP.columns = CKG_P_BP.columns.str.lower()
CKG_P_BP.rename(columns={'head_alt_name': 'head_detail_name', 'tail_go_name': 'tail_detail_name', 'kgsource': 'kg_source'}, inplace=True)
print(f"CKG:      {len(CKG_P_BP):,} rows")

CKG_P_BP['kg_type'] = 'Generalised'
CKG_P_BP['kg_source'] = 'CKG'
CKG_P_BP['species'] = 'Homo species'
CKG_P_BP.head(2)

CKG:      369,710 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,head_id_is,tail_id_is,evidence,score,tail_detail_name,kg_type,species
0,Q96SQ7,ASSOCIATED_WITH,GO:0045603,Protein,Ontology,UniProt,CKG,Transcription factor ATOH8 {ECO:0000305},Uniprot,QuickGO,IDA,5,positive regulation of endothelial cell differ...,Generalised,Homo species
1,P21359,ASSOCIATED_WITH,GO:0021897,Protein,Ontology,UniProt,CKG,Neurofibromin truncated,Uniprot,QuickGO,ISS,5,forebrain astrocyte development,Generalised,Homo species


### 2b. CrossBAR

In [4]:
#

In [5]:
CrossBAR_Protein_BP = pd.read_csv(PROC_DIR + 'crossbar/Protein_BiologicalProcess_Crossbar.csv')
CrossBAR_Protein_BP.columns = CrossBAR_Protein_BP.columns.str.lower()
print(f"CrossBAR: {len(CrossBAR_Protein_BP):,} rows")

CrossBAR_Protein_BP['kg_type'] = 'Generalised'
CrossBAR_Protein_BP['kg_source'] = 'CROssBAR'
CrossBAR_Protein_BP['species'] = 'Homo species'
CrossBAR_Protein_BP.head(2)

CrossBAR: 1,235,325 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,A1DG68,Protein_BiologicalProcess,GO:0006508,Protein,BiologicalProcess,CROssBAR,Leucine aminopeptidase 2,proteolysis,Uniprot_ID,QuickGO,Generalised,Homo species
1,A1DG68,Protein_BiologicalProcess,GO:0043171,Protein,BiologicalProcess,CROssBAR,Leucine aminopeptidase 2,peptide catabolic process,Uniprot_ID,QuickGO,Generalised,Homo species


### 2c. TARKG

In [6]:
TARKG_Protein_BP = pd.read_csv(PROC_DIR + 'TARKG/Protein_BiologicalProcess_TARKG.csv')
TARKG_Protein_BP.columns = TARKG_Protein_BP.columns.str.lower()
print(f"TARKG:    {len(TARKG_Protein_BP):,} rows")
TARKG_Protein_BP['kg_type'] = 'Generalised'
TARKG_Protein_BP['kg_source'] = 'TARKG'
TARKG_Protein_BP['species'] = 'Homo species'

TARKG_Protein_BP.head(2)

TARKG:    537,434 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,Q93YU8,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,Nitrate regulatory gene2 protein {ECO:0000303|...,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-2866080,BioKG,0,Generalised,Homo species
1,P0ABD0,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,High-affinity choline transport protein {ECO:0...,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-2586045,BioKG,0,Generalised,Homo species


### 2d. Monarch

In [7]:
Monarch_Protein_BP = pd.read_csv(PROC_DIR + 'Monarch/Human/Protein_BiologicalProcess_Monarch.csv')
Monarch_Protein_BP.columns = Monarch_Protein_BP.columns.str.lower()
print(f"Monarch:    {len(Monarch_Protein_BP):,} rows")
TARKG_Protein_BP['kg_type'] = 'Generalised'
TARKG_Protein_BP['kg_source'] = 'MonarchKG'
TARKG_Protein_BP['species'] = 'Homo species'
Monarch_Protein_BP.head(2)

Monarch:    391 rows


,head,tail,relation_type,relation_source,kgsource,head_name,head_type,head_id_is,head_taxon,head_taxon_label,...,tail_type,tail_id_is,tail_taxon,tail_taxon_label,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species
0,PR:000000001,GO:0006412,related_to,infores:pr,MonarchKG,protein,Protein,PR,NaN,NaN,...,BiologicalProcess,GO,NaN,NaN,NaN,NaN,Protein,BiologicalProcess,Protein_BiologicalProcess,NaN
1,PR:000000547,GO:0016485,related_to,infores:pr,MonarchKG,tumor necrosis factor ligand superfamily membe...,Protein,PR,NaN,NaN,...,BiologicalProcess,GO,NaN,NaN,NaN,NaN,Protein,BiologicalProcess,Protein_BiologicalProcess,NaN


## 3. Check and Fix Duplicate Columns

In [8]:
SOURCE_DFS = [
    ('CKG_P_BP',            CKG_P_BP),
    ('CrossBAR_Protein_BP', CrossBAR_Protein_BP),
    ('TARKG_Protein_BP',    TARKG_Protein_BP),
    ('Monarch_Protein_BP', Monarch_Protein_BP),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_P_BP] ✓ no duplicates
[CrossBAR_Protein_BP] ✓ no duplicates
[TARKG_Protein_BP] ✓ no duplicates
[Monarch_Protein_BP] ✓ no duplicates


## 4. Align Schemas and Concatenate



In [9]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    tmp['head_id_is'] = tmp['head_id_is'].astype(str)
    tmp['tail_id_is'] = tmp['tail_id_is'].astype(str)
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 2,142,860 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q96SQ7,ASSOCIATED_WITH,GO:0045603,Protein,NaN,Ontology,CKG,Uniprot,QuickGO,Transcription factor ATOH8 {ECO:0000305},positive regulation of endothelial cell differ...,Generalised,Homo species
1,P21359,ASSOCIATED_WITH,GO:0021897,Protein,NaN,Ontology,CKG,Uniprot,QuickGO,Neurofibromin truncated,forebrain astrocyte development,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [10]:
final_df['head_type']  = 'Protein'
final_df['tail_type']  = 'BiologicalProcess'
final_df['relation']   = 'Protein_BiologicalProcess'
final_df['head_id_is'] = 'Uniprot_ID'
final_df['tail_id_is'] = 'Quick_GO'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Protein_BiologicalProcess'}
Unique kg_source: {'MonarchKG', nan, 'CROssBAR', 'CKG'}


## 6. Deduplicate

In [11]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 1,329,453 rows


## 7. Fill Missing Protein Head Names from UniProt

In [12]:
# Normalise fake NaNs to real np.nan
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].replace(['NAN', 'NaN', 'None'], np.nan)

# Fill from UniProt AC→Name dict
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].fillna(
    final_df_dedup['head'].map(uniprot_trembl_AC_Name_dict)
)

# Drop rows still missing a head name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name → {len(final_df_dedup):,} remaining")

Dropped 728,382 rows with missing head_detail_name → 601,071 remaining


## 8. Add Schema Columns and Enforce Column Order

In [13]:
final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (601071, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A024RBG1,Protein_BiologicalProcess,GO:0071543,Protein,None,BiologicalProcess,CKG::CROssBAR,Uniprot_ID,Quick_GO,Diphosphoinositol polyphosphate phosphohydrola...,diphosphoinositol polyphosphate metabolic process,Generalised,Homo species
1,A0A024RBG1,Protein_BiologicalProcess,GO:1901907,Protein,None,BiologicalProcess,CKG::CROssBAR,Uniprot_ID,Quick_GO,Diphosphoinositol polyphosphate phosphohydrola...,diadenosine pentaphosphate catabolic process,Generalised,Homo species
2,A0A024RBG1,Protein_BiologicalProcess,GO:1901909,Protein,None,BiologicalProcess,CKG::CROssBAR,Uniprot_ID,Quick_GO,Diphosphoinositol polyphosphate phosphohydrola...,diadenosine hexaphosphate catabolic process,Generalised,Homo species
3,A0A024RBG1,Protein_BiologicalProcess,GO:1901911,Protein,None,BiologicalProcess,CKG::CROssBAR,Uniprot_ID,Quick_GO,Diphosphoinositol polyphosphate phosphohydrola...,adenosine 5'-(hexahydrogen pentaphosphate) cat...,Generalised,Homo species
4,A0A075B6H5,Protein_BiologicalProcess,GO:0007166,Protein,None,BiologicalProcess,CKG,Uniprot_ID,Quick_GO,A0A075B6H5,cell surface receptor signaling pathway,Generalised,Homo species


In [14]:
final_df_dedup['species'] = 'Homo sapiens'

In [15]:
set(final_df_dedup['relation']), set(final_df_dedup['head_type']),set(final_df_dedup['tail_type']),set(final_df_dedup['kg_source']),set(final_df_dedup['head_id_is']), set(final_df_dedup['tail_id_is']), set(final_df_dedup['kg_type'])

({'Protein_BiologicalProcess'},
 {'Protein'},
 {'BiologicalProcess'},
 {'CKG',
  'CKG::CROssBAR',
  'CKG::CROssBAR::MonarchKG',
  'CKG::MonarchKG',
  'CROssBAR::MonarchKG',
  'MonarchKG'},
 {'Uniprot_ID'},
 {'Quick_GO'},
 {'Generalised'})

## 9. QC — NaN Check

In [16]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,63637,0,63637
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [17]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 601,071 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_BIOLOGICALPROCESS/ALL_PROTEIN_BIOLOGICALPROCESS.parquet


In [18]:
#